# AIBackends - custom tasks, custom pipelines, and batch runs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/AIBackends-custom-tasks-pipelines-batch.ipynb)

Extend aibackends with your own building blocks: register a custom `BaseTask`, assemble a
`Pipeline` from ingest / enrich / validate steps, and fan a workflow out over many inputs
with `run_batch` or `run_async`. Covers `examples/tasks/task_interface.py`,
`examples/workflows/custom_pipeline.py`, and `examples/workflows/batch_processing.py`.

**Runtime:** works on a CPU runtime; for faster inference pick *Runtime > Change runtime type > T4 GPU*. The device is detected automatically.

In [ ]:
import shutil
import subprocess

# Prebuilt llama-cpp-python wheels: CUDA 12.4 build on GPU runtimes, CPU build otherwise.
HAS_NVIDIA_GPU = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
)
LLAMA_WHEEL = (
    "https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.35-cu124/llama_cpp_python-0.3.35-py3-none-manylinux_2_35_x86_64.whl"
    if HAS_NVIDIA_GPU
    else "https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.35/llama_cpp_python-0.3.35-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl"
)
print("GPU runtime:", HAS_NVIDIA_GPU)
print("llama-cpp-python wheel:", LLAMA_WHEEL.rsplit("/", 2)[-2])

%pip install -q "{LLAMA_WHEEL}"
%pip install -q "aibackends[pii]>=0.8.1" huggingface_hub

In [2]:
import shutil
import subprocess

import aibackends
import llama_cpp

# "gpu" offloads every layer to CUDA (n_gpu_layers=-1); "cpu" keeps everything on CPU.
HAS_NVIDIA_GPU = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
)
DEVICE = "gpu" if HAS_NVIDIA_GPU and llama_cpp.llama_supports_gpu_offload() else "cpu"

print("aibackends", aibackends.__version__)
print("llama-cpp-python", llama_cpp.__version__)
print("device:", DEVICE)

aibackends 0.8.1
llama-cpp-python 0.3.35
device: cpu


In [3]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = "https://raw.githubusercontent.com/donvito/aibackends/main/examples/data"
DATA_DIR = Path("aibackends_data")


def fetch(relative_path: str) -> Path:
    """Download a sample file from the aibackends examples once and return its path."""
    path = DATA_DIR / relative_path
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        partial = path.with_name(path.name + ".part")
        try:
            urlretrieve(f"{DATA_URL}/{relative_path}", partial)
            partial.replace(path)
        finally:
            partial.unlink(missing_ok=True)
    return path

## 1. A custom task (`task_interface.py`)

Subclass `BaseTask`, register it, and look it up by name like a built-in. No model needed.

In [4]:
from aibackends.core.registry import TaskSpec
from aibackends.tasks import BaseTask, create_task, get_task, list_tasks, register_task


class WordCountTask(BaseTask):
    name = "word-count"

    def run(self, input: str, **kwargs):
        options = self._resolve_options(**kwargs)
        if options.get("strip", False):
            input = input.strip()
        return {"characters": len(input), "words": len(input.split())}


register_task(TaskSpec(name=WordCountTask.name, task_factory=WordCountTask))
task = create_task(get_task("word-count"), strip=True)
print(task.run("  AIBackends tasks can be objects too.  "))
print("override per call:", task.run("  padded  ", strip=False))
print("word-count registered:", "word-count" in list_tasks())

{'characters': 36, 'words': 6}
override per call: {'characters': 10, 'words': 1}
word-count registered: True


## 2. A custom pipeline (`custom_pipeline.py`)

`FileIngestor` reads the note, `LLMAnalyser` extracts a schema with Gemma 4 E2B on
llama.cpp, and `PydanticValidator` guarantees the output type. `device=DEVICE` flows into
every LLM step of the pipeline.

In [5]:
from pydantic import BaseModel

from aibackends.models import GEMMA4_E2B
from aibackends.runtimes import LLAMACPP
from aibackends.steps.enrich import LLMAnalyser
from aibackends.steps.ingest import FileIngestor
from aibackends.steps.validate import PydanticValidator
from aibackends.workflows import Pipeline

LLM = {"runtime": LLAMACPP, "model": GEMMA4_E2B, "device": DEVICE}


class LeadBrief(BaseModel):
    name: str
    company: str | None = None
    email: str | None = None
    priority: str | None = None
    next_step: str | None = None


class LeadIntakePipeline(Pipeline):
    steps = [
        FileIngestor(),
        LLMAnalyser(
            schema=LeadBrief,
            prompt=(
                "Extract the lead details from the note: name, company, email, priority "
                "(low, medium, or high), and next_step. Use null when a detail is missing."
            ),
        ),
        PydanticValidator(schema=LeadBrief),
    ]


lead_pipeline = LeadIntakePipeline(**LLM)
print(lead_pipeline.run(fetch("lead_note.txt")).model_dump_json(indent=2))

{
  "name": "Priya Nair",
  "company": "Acme Retail",
  "email": "priya@acmeretail.com",
  "priority": "high",
  "next_step": "send a proposal and timeline by Friday"
}


## 3. Batch processing (`batch_processing.py`)

`run_batch` runs a workflow over many inputs with bounded concurrency; `on_error="collect"`
returns failures alongside results instead of raising. This pipeline mirrors the
built-in `SalesCallAnalyser` (ingest -> transcribe if audio -> redact PII -> analyse ->
validate) with explicit field guidance for small local models. Swap in
`create_workflow(SalesCallAnalyser, **LLM)` to use the built-in directly.

In [6]:
import time

from aibackends.schemas.sales_call import SalesCallReport
from aibackends.steps.enrich import PIIRedactor
from aibackends.steps.ingest import AudioIngestor
from aibackends.steps.process import WhisperTranscriber
from aibackends.workflows import SalesCallAnalyser, list_workflows

print("built-in workflows:", list_workflows())


class SalesCallBatchPipeline(Pipeline):
    steps = [
        AudioIngestor(),
        WhisperTranscriber(),
        PIIRedactor(backend="gliner"),
        LLMAnalyser(
            schema=SalesCallReport,
            prompt=(
                "Analyse the sales call transcript. Return talk_ratio as an object with agent "
                'and customer shares that sum to 1, e.g. {"agent": 0.55, "customer": 0.45}; '
                "objections, buying_signals, and action_items as lists of short strings; score "
                "as a number from 0 to 10; sentiment as one word."
            ),
        ),
        PydanticValidator(schema=SalesCallReport),
    ]


transcripts = [fetch("batch/sales_call_1.txt"), fetch("batch/sales_call_2.txt")]
t = time.perf_counter()
batch = SalesCallBatchPipeline(**LLM).run_batch(inputs=transcripts, max_concurrency=2, on_error="collect")
print(f"{len(batch.results)} results, {len(batch.errors)} errors in {time.perf_counter() - t:.0f}s")
print(batch.model_dump_json(indent=2))

built-in workflows: ['embedding-similarity', 'invoice', 'pii-redactor', 'sales-call', 'video-ad']


2 results, 0 errors in 29s
{
  "results": [
    {
      "talk_ratio": {
        "agent": 0.5,
        "customer": 0.5
      },
      "objections": [
        "Procurement pilot request"
      ],
      "buying_signals": [
        "Team likes it",
        "Ready to move quickly next month"
      ],
      "action_items": [
        "Propose 50 documents pilot",
        "Set two-week review period"
      ],
      "score": 8.0,
      "sentiment": "Positive"
    },
    {
      "talk_ratio": {
        "agent": 0.66,
        "customer": 0.34
      },
      "objections": [
        "Budget concern"
      ],
      "buying_signals": [
        "Likes on-prem setup"
      ],
      "action_items": [
        "Send phase one scope",
        "Wait for legal sign-off"
      ],
      "score": 7.0,
      "sentiment": "Positive"
    }
  ],
  "errors": []
}


## 4. Async

Every pipeline has `run_async`, so it slots into async web handlers.

In [7]:
import asyncio

briefs = await asyncio.gather(
    lead_pipeline.run_async(
        "New lead: Maria Chen, Head of Ops at Northwind (maria@northwind.io). "
        "Medium priority. Next step: book a demo for next Tuesday."
    ),
    lead_pipeline.run_async(
        "Tom Reyes, CTO at Globex, emailed from tom@globex.com asking for enterprise pricing. "
        "High priority. Next step: send the pricing sheet today."
    ),
)
for brief in briefs:
    print(brief.model_dump_json())

{"name":"Maria Chen","company":"Northwind","email":"maria@northwind.io","priority":"medium","next_step":"book a demo for next Tuesday"}
{"name":"Tom Reyes","company":"Globex","email":"tom@globex.com","priority":"high","next_step":"send the pricing sheet today"}
